# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant metadata. We'll explore all record sets and print record samples and field/column structure using their `@id` identifiers.

In [ ]:
# List all record sets in the dataset and their fields/columns (@id references)
record_set_ids = []
print("Available Record Sets (by @id):\n")
for rs in metadata.record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")
    record_set_ids.append(rs['@id'])
    # List fields or columns
    print("    Fields (by @id):")
    for field in rs.get('fields', []):
        print(f"      - {field['@id']} (name: {field.get('name', 'N/A')}, type: {field.get('dataType', 'N/A')})")
    for col in rs.get('columns', []):
        print(f"      - {col['@id']} (column: {col.get('name', 'N/A')}, type: {col.get('dataType', 'N/A')})")
    print()
# As a sample, print a few records (if available)
if record_set_ids:
    sample_id = record_set_ids[0]
    print(f"\nFirst 2 sample records in {sample_id}:")
    for i, rec in enumerate(dataset.records(record_set=sample_id)):
        print(rec)
        if i == 1:
            break
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above. We extract each available record set to its own DataFrame, referenced by their record set `@id`.

In [ ]:
# Extract data from each record set into individual DataFrames
dataframes = {}
# Use list of record sets from previous cell
for record_set_id in record_set_ids:
    print(f"Loading records from {record_set_id} ...")
    # `records` returns a generator of dict
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows. Columns: {df.columns.tolist()}")

# Preview the first record set (if available)
if record_set_ids:
    preview_id = record_set_ids[0]
    print(f"\nPreview of DataFrame for record set {preview_id}:")
    display(dataframes[preview_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Note:** Update the field `@id`s according to the printed overview above.

In [ ]:
# For demonstration, we'll select a numeric field and perform filtering & normalization.
# These field names/@ids should be updated as per dataset's structure. Here, we try common ones for proteomics tables.
import numpy as np

chosen_record_set = None
chosen_numeric_field = None
group_field = None

# Find a record set with a numeric field automatically for demo purposes
for rsid, df in dataframes.items():
    numeric_candidates = [c for c in df.columns if df[c].dtype in (np.float64, np.float32, np.int64, np.int32)]
    if len(numeric_candidates) > 0:
        chosen_record_set = rsid
        chosen_numeric_field = numeric_candidates[0]
        # Pick a non-numeric field to group if possible
        non_numeric = [c for c in df.columns if df[c].dtype == object and c != chosen_numeric_field]
        if non_numeric:
            group_field = non_numeric[0]
        break

if chosen_record_set and chosen_numeric_field:
    print(f"Using record set: {chosen_record_set}")
    print(f"Using numeric field (for filter/normalize): {chosen_numeric_field}")
    if group_field:
        print(f"Grouping by: {group_field}")
    df = dataframes[chosen_record_set]

    threshold = df[chosen_numeric_field].mean() if df[chosen_numeric_field].dtype != object else 0
    # Ensure numeric
    df[chosen_numeric_field] = pd.to_numeric(df[chosen_numeric_field], errors='coerce')

    filtered_df = df[df[chosen_numeric_field] > threshold]
    print(f"Filtered records with {chosen_numeric_field} > {threshold:.3f} (n={len(filtered_df)}):")
    display(filtered_df.head())

    filtered_df[f"{chosen_numeric_field}_normalized"] = (filtered_df[chosen_numeric_field] - filtered_df[chosen_numeric_field].mean()) / filtered_df[chosen_numeric_field].std()
    print(f"Normalized {chosen_numeric_field} for filtered records:")
    display(filtered_df[[chosen_numeric_field, f"{chosen_numeric_field}_normalized"]].head())

    if group_field and group_field in filtered_df.columns:
        # Group and compute average of numeric field
        grouped_df = filtered_df.groupby(group_field)[chosen_numeric_field].mean().reset_index()
        print(f"Grouped average of {chosen_numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA. Please update field names as appropriate.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set and chosen_numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=dataframes[chosen_record_set], x=chosen_numeric_field, kde=True)
    plt.title(f"Distribution of {chosen_numeric_field} in {chosen_record_set}")
    plt.xlabel(chosen_numeric_field)
    plt.show()

    if group_field and group_field in dataframes[chosen_record_set].columns:
        plt.figure(figsize=(8,4))
        # Boxplot across groups
        sns.boxplot(data=dataframes[chosen_record_set], x=group_field, y=chosen_numeric_field)
        plt.title(f"{chosen_numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset _Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells_ using the `mlcroissant` library. We inspected the available record sets, fields, and columns by their `@id`, extracted the data, performed basic EDA, normalization, and grouped analysis, and visualized key distributions.

This is a starting point for advanced data mining, statistical analysis, or machine learning using FAIR Croissant-described data packages.